In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path("..") / "data" / "raw"

meta = pd.read_csv(RAW / "_metadata.csv")
print(meta.shape)
meta

(29, 6)


,symbol,yf_ticker,sector,rows,first_date,last_date
0,NIFTY50,^NSEI,Index,1236,2021-10-18,2026-10-16
1,TCS,TCS.NS,IT,1237,2021-10-18,2026-10-16
2,INFY,INFY.NS,IT,1237,2021-10-18,2026-10-16
3,WIPRO,WIPRO.NS,IT,1237,2021-10-18,2026-10-16
4,HCLTECH,HCLTECH.NS,IT,1237,2021-10-18,2026-10-16
5,HDFCBANK,HDFCBANK.NS,Banking,1237,2021-10-18,2026-10-16
6,ICICIBANK,ICICIBANK.NS,Banking,1237,2021-10-18,2026-10-16
7,SBIN,SBIN.NS,Banking,1237,2021-10-18,2026-10-16
8,AXISBANK,AXISBANK.NS,Banking,1237,2021-10-18,2026-10-16
9,KOTAKBANK,KOTAKBANK.NS,Banking,1237,2021-10-18,2026-10-16


In [2]:
tcs = pd.read_csv(RAW / "TCS.csv", parse_dates=["Date"])
print(tcs.shape)
print(tcs.dtypes)
tcs.head()

(1237, 6)
Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object


,Date,Close,High,Low,Open,Volume
0,2021-10-18,3157.567139,3162.372165,3118.001816,3119.733341,3590107
1,2021-10-19,3146.311768,3194.751365,3138.563233,3176.483685,3135441
2,2021-10-20,3124.408691,3142.719493,3097.959591,3127.135760,2933693
3,2021-10-21,3058.307617,3123.196508,3039.910155,3120.296329,4871211
4,2021-10-22,3029.174561,3083.501182,3017.183660,3082.115877,2691670


In [3]:
close = {}
for sym in meta["symbol"]:
    df = pd.read_csv(RAW / f"{sym}.csv", parse_dates=["Date"])
    close[sym] = df.set_index("Date")["Close"]

prices = pd.DataFrame(close)
print(prices.shape)
prices.tail()

(1237, 29)


,NIFTY50,TCS,INFY,WIPRO,HCLTECH,HDFCBANK,ICICIBANK,SBIN,AXISBANK,KOTAKBANK,...,RELIANCE,ONGC,NTPC,POWERGRID,TATASTEEL,HINDALCO,LT,ASIANPAINT,ULTRACEMCO,BHARTIARTL
Date,,,,,,,,,,,,,,,,,,,,,
2026-10-12,22363.349609,2113.763184,1020.000000,161.800003,1212.208862,704.900024,1340.000000,941.000000,1253.000000,443.399994,...,1158.900024,220.000000,310.000000,249.000000,172.240005,882.500000,3627.500000,2331.000000,10520.0,1778.699951
2026-10-13,22721.949219,2257.300049,1062.099976,169.000000,1314.300049,714.000000,1360.199951,957.000000,1268.400024,439.000000,...,1166.699951,222.649994,311.000000,250.500000,175.000000,902.250000,3675.000000,2376.000000,10750.0,1806.500000
2026-10-14,22622.699219,2304.899902,1088.000000,170.919998,1315.291138,710.700012,1338.599976,954.900024,1264.599976,438.049988,...,1174.599976,225.500000,313.700012,248.949997,174.490005,882.500000,3642.000000,2344.699951,10795.0,1794.599976
2026-10-15,22997.349609,2350.000000,1110.000000,174.050003,1348.000000,726.849976,1389.400024,960.000000,1288.000000,447.950012,...,1215.000000,224.639999,314.649994,249.850006,176.399994,893.849976,3632.000000,2350.000000,10862.0,1854.000000
2026-10-16,23146.150391,2465.000000,1161.199951,184.399994,1380.000000,728.849976,1397.000000,962.599976,1290.000000,447.399994,...,1211.199951,222.000000,315.149994,250.449997,175.500000,908.000000,3627.100098,2331.899902,10907.0,1905.000000


In [4]:
print("Missing values per column (only those with any):")
missing = prices.isna().sum()
print(missing[missing > 0])

print("\nDates with at least one missing value:")
prices[prices.isna().any(axis=1)]

Missing values per column (only those with any):
NIFTY50    1
dtype: int64

Dates with at least one missing value:


,NIFTY50,TCS,INFY,WIPRO,HCLTECH,HDFCBANK,ICICIBANK,SBIN,AXISBANK,KOTAKBANK,...,RELIANCE,ONGC,NTPC,POWERGRID,TATASTEEL,HINDALCO,LT,ASIANPAINT,ULTRACEMCO,BHARTIARTL
Date,,,,,,,,,,,,,,,,,,,,,
2026-01-15,NaN,3060.415527,1565.914551,250.78717,1594.759399,910.392639,1406.542114,1010.14209,1297.799194,420.274414,...,1452.086182,241.495392,342.78125,254.296692,185.425995,950.392029,3828.406738,2789.696533,12009.859375,1997.362671


In [5]:
fake = {}
for sym in meta["symbol"]:
    if sym == "NIFTY50":
        continue
    df = pd.read_csv(RAW / f"{sym}.csv", parse_dates=["Date"])
    flat = (df.Volume == 0) & (df.Open == df.High) & (df.High == df.Low) & (df.Low == df.Close)
    fake[sym] = df.loc[flat, "Date"].dt.date.tolist()

pd.Series({s: len(d) for s, d in fake.items()}).value_counts()

2    25
3     2
1     1
Name: count, dtype: int64